In [ ]:
print("[cell] setup", flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || true")
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count(), flush=True)
PIP = os.environ.get("PIP_SPEC", 'vllm==0.32.0 peft')   # --env PIP_SPEC=... / PIP_POST=... override the built-in install
if PIP and torch.cuda.is_available():
    sh(f"pip install -q {PIP}")
    if os.environ.get("PIP_POST", 'pip uninstall -y torchaudio torchao'):
        sh(os.environ.get("PIP_POST", 'pip uninstall -y torchaudio torchao'))
sh("pip list 2>/dev/null | grep -i -E '^(vllm|torch|transformers|qwen-asr|pandas|openpyxl) ' || true")


In [ ]:
print("[cell] write code files", flush=True)
FILES = {
    'common.py': '"""Shared helpers for GreenScan training on Kaggle (T4) and on a rented A100 (vast.ai).\n\nData is looked up in $GS_DATA, ./data, ../data, then /kaggle/input (a private dataset). Outputs go to $GS_OUT\n(default ./out). Every script prints "[cell] ..." at each stage and flushes, so a log shows where a run is.\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport random\nimport re\nimport time\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nimport numpy as np\n\n\ndef cell(name: str) -> None:\n    print(f"[cell] {name}", flush=True)\n\n\ndef log(*a) -> None:\n    print(*a, flush=True)\n\n\ndef find(name: str) -> Path:\n    for root in (os.environ.get("GS_DATA", ""), "data", "../data", "/kaggle/input"):\n        if root and Path(root).exists():\n            hit = next(iter(sorted(Path(root).rglob(name))), None)\n            if hit:\n                return hit\n    raise FileNotFoundError(name)\n\n\ndef has(name: str) -> bool:\n    try:\n        find(name)\n        return True\n    except FileNotFoundError:\n        return False\n\n\ndef jl(path) -> list[dict]:\n    p = Path(path) if not isinstance(path, str) or Path(path).exists() else find(path)\n    return [json.loads(x) for x in open(p, encoding="utf-8") if x.strip()]\n\n\ndef wjl(path, rows) -> None:\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    with open(path, "w", encoding="utf-8") as fh:\n        for r in rows:\n            fh.write(json.dumps(r, ensure_ascii=False) + "\\n")\n\n\ndef out_dir(sub: str = "") -> Path:\n    p = Path(os.environ.get("GS_OUT", "out")) / sub\n    p.mkdir(parents=True, exist_ok=True)\n    return p\n\n\ndef save_json(path, obj) -> None:\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=1), encoding="utf-8")\n\n\ndef seed_all(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    try:\n        import torch\n        torch.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n    except ImportError:\n        pass\n\n\ndef norm60(s: str) -> str:\n    return re.sub(r"\\s+", " ", (s or "").lower()).strip()[:60]\n\n\ndef company_folds(groups: list[str], k: int = 5, seed: int = 13) -> list[tuple[list[int], list[int]]]:\n    """Out-of-company folds: every company sits in exactly one test fold; folds balanced by row count."""\n    count = Counter(groups)\n    comps = sorted(count, key=lambda c: (-count[c], c))\n    rng = random.Random(seed)\n    rng.shuffle(comps)\n    comps.sort(key=lambda c: -count[c])\n    load, fold_of = [0] * k, {}\n    for c in comps:\n        f = min(range(k), key=lambda i: load[i])\n        fold_of[c] = f\n        load[f] += count[c]\n    out = []\n    for f in range(k):\n        test = [i for i, g in enumerate(groups) if fold_of[g] == f]\n        train = [i for i, g in enumerate(groups) if fold_of[g] != f]\n        if test and train:\n            out.append((train, test))\n    return out\n\n\ndef prf(gold: list[str], pred: list[str], labels: list[str]) -> dict:\n    per = {}\n    for l in labels:\n        tp = sum(g == l and p == l for g, p in zip(gold, pred))\n        fp = sum(g != l and p == l for g, p in zip(gold, pred))\n        fn = sum(g == l and p != l for g, p in zip(gold, pred))\n        pr = tp / (tp + fp) if tp + fp else 0.0\n        rc = tp / (tp + fn) if tp + fn else 0.0\n        per[l] = {"precision": round(pr, 4), "recall": round(rc, 4),\n                  "f1": round(2 * pr * rc / (pr + rc), 4) if pr + rc else 0.0, "support": tp + fn, "predicted": tp + fp}\n    present = [l for l in labels if per[l]["support"]]\n    macro = sum(per[l]["f1"] for l in present) / len(present) if present else 0.0\n    acc = sum(g == p for g, p in zip(gold, pred)) / len(gold) if gold else 0.0\n    return {"macro_f1": round(macro, 4), "accuracy": round(acc, 4), "per_class": per, "n": len(gold)}\n\n\ndef confusion(gold: list[str], pred: list[str], labels: list[str]) -> list[list[int]]:\n    idx = {l: i for i, l in enumerate(labels)}\n    m = [[0] * len(labels) for _ in labels]\n    for g, p in zip(gold, pred):\n        if g in idx and p in idx:\n            m[idx[g]][idx[p]] += 1\n    return m\n\n\ndef boot_ci(items: list, groups: list[str], stat, n: int = 1000, seed: int = 7) -> list[float] | None:\n    """95% interval of stat(items) resampling whole companies (cluster bootstrap)."""\n    by = defaultdict(list)\n    for it, g in zip(items, groups):\n        by[g].append(it)\n    keys = sorted(by)\n    if len(keys) < 2:\n        return None\n    rng = random.Random(seed)\n    vals = []\n    for _ in range(n):\n        sample = [it for _ in keys for it in by[rng.choice(keys)]]\n        try:\n            vals.append(stat(sample))\n        except ZeroDivisionError:\n            continue\n    vals.sort()\n    return [round(vals[int(0.025 * len(vals))], 4), round(vals[int(0.975 * len(vals)) - 1], 4)] if vals else None\n\n\nclass Timer:\n    def __init__(self):\n        self.t0 = time.time()\n\n    def __call__(self) -> float:\n        return round(time.time() - self.t0, 1)\n',
    'router_metrics.py': '"""Metrics of prompt_router_v1, shared by the gate, the planner and the rules baseline.\n\nA prediction is {"in_scope": bool, "tasks": [...], "slots": {...}} (or None when unparsed: counted as a refusal, the\nsafe side). Decision of a row: out_of_scope / need_info / ok, with need_info = in scope and a claim-needing task without\na claim. OOS recall = out-of-scope rows predicted out (the safety number); false refusal = in-scope rows predicted out.\n"""\nfrom __future__ import annotations\n\nimport random\n\nTASKS = ["verify_claim", "extract_claims", "check_attributes", "explain_verdict", "find_evidence", "review_assurance",\n         "assess_report_risk"]\nNEEDS_CLAIM = {"verify_claim", "check_attributes", "explain_verdict", "find_evidence"}\n\n\ndef decision(p: dict | None) -> str:\n    if not p or not p.get("in_scope"):\n        return "out_of_scope"\n    if any(t in NEEDS_CLAIM for t in p.get("tasks") or []) and not (p.get("slots") or {}).get("claim"):\n        return "need_info"\n    return "ok"\n\n\ndef _ci(flags: list[bool], groups: list[str], n: int = 1000, seed: int = 7) -> list[float] | None:\n    by: dict[str, list[bool]] = {}\n    for f, g in zip(flags, groups):\n        by.setdefault(g, []).append(f)\n    keys = sorted(by)\n    if len(keys) < 2:\n        return None\n    rng, vals = random.Random(seed), []\n    for _ in range(n):\n        s = [x for _ in keys for x in by[rng.choice(keys)]]\n        vals.append(sum(s) / len(s))\n    vals.sort()\n    return [round(vals[25], 4), round(vals[974], 4)]\n\n\ndef score(gold: list[dict], pred: list[dict | None]) -> dict:\n    oos = [(g, p) for g, p in zip(gold, pred) if not g["in_scope"]]\n    ins = [(g, p) for g, p in zip(gold, pred) if g["in_scope"]]\n    caught = [decision(p) == "out_of_scope" for _, p in oos]\n    refused = [decision(p) == "out_of_scope" for _, p in ins]\n    grp = lambda g: g.get("oos_category") or (g.get("slots") or {}).get("company") or "-"   # noqa: E731\n    out = {"n": len(gold), "n_out": len(oos), "n_in": len(ins),\n           "oos_recall": round(sum(caught) / max(1, len(caught)), 4), "oos_recall_ci95": _ci(caught, [grp(g) for g, _ in oos]),\n           "false_refusal": round(sum(refused) / max(1, len(refused)), 4),\n           "false_refusal_ci95": _ci(refused, [grp(g) for g, _ in ins]),\n           "decision_accuracy": round(sum(decision(g) == decision(p) for g, p in zip(gold, pred)) / max(1, len(gold)), 4),\n           "unparsed": sum(p is None for p in pred),\n           "oos_missed_by_category": {}}\n    for g, p in oos:\n        if decision(p) != "out_of_scope":\n            c = g["oos_category"]\n            out["oos_missed_by_category"][c] = out["oos_missed_by_category"].get(c, 0) + 1\n    f1s, tp_all, fp_all, fn_all = {}, 0, 0, 0\n    for t in TASKS:\n        tp = sum(t in (g.get("tasks") or []) and t in ((p or {}).get("tasks") or []) for g, p in ins)\n        fp = sum(t not in (g.get("tasks") or []) and t in ((p or {}).get("tasks") or []) for g, p in ins)\n        fn = sum(t in (g.get("tasks") or []) and t not in ((p or {}).get("tasks") or []) for g, p in ins)\n        tp_all, fp_all, fn_all = tp_all + tp, fp_all + fp, fn_all + fn\n        pr, rc = tp / (tp + fp) if tp + fp else 0.0, tp / (tp + fn) if tp + fn else 0.0\n        f1s[t] = round(2 * pr * rc / (pr + rc), 4) if pr + rc else 0.0\n    out["task_f1"] = f1s\n    out["task_macro_f1"] = round(sum(f1s.values()) / len(f1s), 4)\n    pr, rc = tp_all / max(1, tp_all + fp_all), tp_all / max(1, tp_all + fn_all)\n    out["task_micro_f1"] = round(2 * pr * rc / (pr + rc), 4) if pr + rc else 0.0\n    for k in ("claim", "company", "year"):\n        rows = [(g, p) for g, p in ins if (g.get("slots") or {}).get(k) is not None]\n        em = [str(((p or {}).get("slots") or {}).get(k) or "").strip().lower() == str(g["slots"][k]).strip().lower()\n              for g, p in rows]\n        out[f"slot_{k}_exact"] = round(sum(em) / max(1, len(em)), 4)\n    return out\n',
    'train_router_gate.py': '"""prompt_router_v1 gate (CPU): multilingual-e5-small with 8 sigmoid outputs (out + 7 tasks), BCE loss.\n\n    python -u train_router_gate.py [--epochs 5]\n\nTrain on router_train, choose the thresholds on router_dev: tau_out is the largest value whose dev OOS recall is still\n>= 0.98 (a request stops when P(out) >= tau_out or no task reaches 0.3), then score router_test once with\nrouter_metrics.score. Writes out/router_gate/metrics.json, history.json and the model (gate-final).\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport math\nimport random\nimport sys\nimport time\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom common import cell, jl, log, out_dir, save_json, seed_all  # noqa: E402\nfrom router_metrics import TASKS, score  # noqa: E402\n\nBASE = "intfloat/multilingual-e5-small"\nOUTS = ["out"] + TASKS\nMAX_LEN, BATCH, TASK_MIN = 192, 32, 0.3\n\n\ndef target(r):\n    return [0.0 if r["in_scope"] else 1.0] + [1.0 if t in (r.get("tasks") or []) else 0.0 for t in TASKS]\n\n\ndef probs(model, tok, texts, dev):\n    import torch\n    out = []\n    with torch.no_grad():\n        for i in range(0, len(texts), 64):\n            x = tok(["query: " + t for t in texts[i:i + 64]], truncation=True, max_length=MAX_LEN, padding=True,\n                    return_tensors="pt").to(dev)\n            out += torch.sigmoid(model(**x).logits.float()).cpu().tolist()\n    return out\n\n\ndef to_pred(p, tau_out):\n    if p[0] >= tau_out or max(p[1:]) < TASK_MIN:\n        return {"in_scope": False, "tasks": [], "slots": {}}\n    return {"in_scope": True, "tasks": [t for t, v in zip(TASKS, p[1:]) if v >= TASK_MIN],\n            "slots": {"claim": "x"}}   # the gate does not extract slots: need_info is the planner\'s / rules\' call\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--epochs", type=int, default=5)\n    ap.add_argument("--lr", type=float, default=5e-5)\n    a = ap.parse_args()\n    import torch\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup\n    seed_all(13)\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    cell("router gate data")\n    tr, dv, te = jl("router_train.jsonl"), jl("router_dev.jsonl"), jl("router_test.jsonl")\n    log(f"[gate] train {len(tr)} dev {len(dv)} test {len(te)}")\n    tok = AutoTokenizer.from_pretrained(BASE)\n    model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=len(OUTS),\n                                                               problem_type="multi_label_classification").float().to(dev)\n    steps = a.epochs * math.ceil(len(tr) / BATCH)\n    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=0.01)\n    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)\n    lossf = torch.nn.BCEWithLogitsLoss()\n    hist = []\n    cell("router gate train")\n    for ep in range(a.epochs):\n        model.train()\n        random.shuffle(tr)\n        tot = 0.0\n        for i in range(0, len(tr), BATCH):\n            b = tr[i:i + BATCH]\n            x = tok(["query: " + r["request"] for r in b], truncation=True, max_length=MAX_LEN, padding=True,\n                    return_tensors="pt").to(dev)\n            loss = lossf(model(**x).logits.float(), torch.tensor([target(r) for r in b], device=dev))\n            opt.zero_grad()\n            loss.backward()\n            opt.step()\n            sch.step()\n            tot += loss.item() * len(b)\n        model.eval()\n        pd = probs(model, tok, [r["request"] for r in dv], dev)\n        dl = -sum(sum(t * math.log(max(p, 1e-7)) + (1 - t) * math.log(max(1 - p, 1e-7)) for t, p in zip(target(r), q)) / len(OUTS)\n                  for r, q in zip(dv, pd)) / max(1, len(dv))\n        hist.append({"run": "final", "epoch": ep + 1, "loss": round(tot / len(tr), 4), "dev_loss": round(dl, 4)})\n        log(f"[gate] epoch {ep + 1} loss {tot / len(tr):.4f} dev {dl:.4f}")\n    cell("router gate thresholds + test")\n    pd = probs(model, tok, [r["request"] for r in dv], dev)\n    cands = sorted({round(x / 100, 2) for x in range(5, 96)}, reverse=True)\n    tau = next((t for t in cands if score(dv, [to_pred(p, t) for p in pd])["oos_recall"] >= 0.98), 0.05)\n    pt = probs(model, tok, [r["request"] for r in te], dev)\n    m = {"tau_out": tau, "task_min": TASK_MIN, "dev": score(dv, [to_pred(p, tau) for p in pd]),\n         "test": score(te, [to_pred(p, tau) for p in pt]), "test_at_0.5": score(te, [to_pred(p, 0.5) for p in pt])}\n    od = out_dir("router_gate")\n    model.save_pretrained(od / "gate-final")\n    tok.save_pretrained(od / "gate-final")\n    cpu = model.to("cpu")\n    torch.set_num_threads(4)\n    t0 = time.time()\n    probs(cpu, tok, [r["request"] for r in te[:64]], "cpu")\n    m["cpu_ms_per_request_4threads"] = round((time.time() - t0) / 64 * 1000, 1)\n    save_json(od / "metrics.json", m)\n    save_json(od / "history.json", hist)\n    t = m["test"]\n    log(f"[gate] tau {tau} · test OOS recall {t[\'oos_recall\']} {t[\'oos_recall_ci95\']} · false refusal {t[\'false_refusal\']} "\n        f"· task macro-F1 {t[\'task_macro_f1\']} · cpu {m[\'cpu_ms_per_request_4threads\']} ms · DONE")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n',
    'train_llm_sft.py': '"""LoRA supervised fine-tuning of a chat model: loss = token cross-entropy on the answer only (prompt masked).\n\n    python -u train_llm_sft.py --base Qwen/Qwen3-4B-Instruct-2507 --train out/teacher/explain_teacher.jsonl \\\n        --name explain-4b --epochs 2 --dtype fp16 [--merge]          # Kaggle T4\n    python -u train_llm_sft.py --base Qwen/Qwen3-14B --train data/stance_sft.jsonl --name stance-14b --dtype bf16  # A100\n\nInput rows: {"id", "company", "messages": [system, user], "target": "<assistant answer>"}. About 10% of companies are\nheld out for the validation loss (out-of-company). Writes out/<name>/adapter (+ merged/ with --merge) and\nout/<name>/history.json (train loss every 10 steps, validation loss every epoch) for the charts.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport math\nimport random\nimport sys\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom common import Timer, cell, jl, log, out_dir, save_json, seed_all  # noqa: E402\n\n\ndef encode(tok, row, max_len):\n    prompt = tok.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=True, enable_thinking=False)\n    p = tok(prompt, add_special_tokens=False)["input_ids"]\n    a = tok(row["target"] + (tok.eos_token or ""), add_special_tokens=False)["input_ids"]\n    if len(p) + len(a) > max_len:\n        return None\n    return {"input_ids": p + a, "labels": [-100] * len(p) + a}\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--base", required=True)\n    ap.add_argument("--train", required=True)\n    ap.add_argument("--name", required=True)\n    ap.add_argument("--epochs", type=float, default=2)\n    ap.add_argument("--lr", type=float, default=2e-4)\n    ap.add_argument("--r", type=int, default=16)\n    ap.add_argument("--alpha", type=int, default=32)\n    ap.add_argument("--max-len", type=int, default=2048)\n    ap.add_argument("--accum", type=int, default=8)\n    ap.add_argument("--dtype", choices=["fp16", "bf16"], default="fp16")\n    ap.add_argument("--val-frac", type=float, default=0.1)\n    ap.add_argument("--merge", action="store_true")\n    ap.add_argument("--merge-dir", default="", help="where the merged fp16 model goes (Kaggle: /tmp, outside the 20 GB output)")\n    ap.add_argument("--seed", type=int, default=13)\n    args = ap.parse_args()\n    seed_all(args.seed)\n\n    cell(f"sft load {args.base}")\n    import torch\n    from peft import LoraConfig, get_peft_model\n    from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup\n    dtype = torch.float16 if args.dtype == "fp16" else torch.bfloat16\n    tok = AutoTokenizer.from_pretrained(args.base)\n    rows = jl(args.train)\n    comps = sorted({r.get("company") or "?" for r in rows})\n    random.Random(args.seed).shuffle(comps)\n    val_comps = set(comps[: max(1, round(len(comps) * args.val_frac))]) if len(comps) > 3 else set()\n    enc = [(r, encode(tok, r, args.max_len)) for r in rows]\n    dropped = sum(e is None for _, e in enc)\n    train = [e for r, e in enc if e and (r.get("company") or "?") not in val_comps]\n    val = [e for r, e in enc if e and (r.get("company") or "?") in val_comps]\n    log(f"[sft] rows {len(rows)} · train {len(train)} · val {len(val)} (companies {sorted(val_comps)}) · "\n        f"too long {dropped}")\n\n    model = AutoModelForCausalLM.from_pretrained(args.base, torch_dtype=dtype, device_map={"": 0})\n    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    model.enable_input_require_grads()\n    model = get_peft_model(model, LoraConfig(r=args.r, lora_alpha=args.alpha, lora_dropout=0.05, task_type="CAUSAL_LM",\n                                             target_modules=["q_proj", "k_proj", "v_proj", "o_proj",\n                                                             "gate_proj", "up_proj", "down_proj"]))\n    model.print_trainable_parameters()\n    params = [p for p in model.parameters() if p.requires_grad]\n    steps = math.ceil(len(train) * args.epochs / args.accum)\n    opt = torch.optim.AdamW(params, lr=args.lr, weight_decay=0.0)\n    sch = get_cosine_schedule_with_warmup(opt, max(1, int(0.05 * steps)), steps)\n    scaler = torch.amp.GradScaler(enabled=args.dtype == "fp16")\n    hist = {"train": [], "val": [], "config": vars(args), "train_rows": len(train), "val_rows": len(val)}\n\n    def loss_of(e):\n        ids = torch.tensor([e["input_ids"]], device="cuda")\n        lab = torch.tensor([e["labels"]], device="cuda")\n        with torch.autocast("cuda", dtype=dtype):\n            return model(input_ids=ids, labels=lab).loss\n\n    def evaluate():\n        if not val:\n            return None\n        model.eval()\n        with torch.no_grad():\n            v = [loss_of(e).float().item() for e in val]\n        model.train()\n        return round(sum(v) / len(v), 4)\n\n    cell("sft train")\n    t, step, seen, run, bad = Timer(), 0, 0, [], 0\n    hist["val"].append({"epoch": 0, "loss": evaluate()})\n    model.train()\n    total = int(len(train) * args.epochs)\n    order = []\n    while len(order) < total:\n        ep = list(range(len(train)))\n        random.shuffle(ep)\n        order += ep\n    for i, j in enumerate(order[:total], 1):\n        loss = loss_of(train[j]) / args.accum\n        if not torch.isfinite(loss):\n            bad += 1\n            if bad >= 5:\n                raise SystemExit("[sft] loss is not finite 5 times: try --dtype bf16 (A100) or a smaller --lr")\n            opt.zero_grad(set_to_none=True)\n            continue\n        scaler.scale(loss).backward()\n        run.append(loss.item() * args.accum)\n        if i % args.accum == 0 or i == total:\n            scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(params, 1.0)\n            scaler.step(opt)\n            scaler.update()\n            opt.zero_grad(set_to_none=True)\n            sch.step()\n            step += 1\n            if step % 10 == 0 or i == total:\n                hist["train"].append({"step": step, "epoch": round(i / len(train), 3), "loss": round(sum(run) / len(run), 4),\n                                      "lr": sch.get_last_lr()[0]})\n                log(f"[sft] step {step}/{steps} epoch {i / len(train):.2f} loss {sum(run) / len(run):.4f} · {t()} s")\n                run = []\n        if i % len(train) == 0 and i < total:\n            v = evaluate()\n            hist["val"].append({"epoch": round(i / len(train), 2), "loss": v})\n            log(f"[sft] epoch {i / len(train):.0f} val loss {v}")\n    hist["val"].append({"epoch": args.epochs, "loss": evaluate()})\n    hist["seconds"] = t()\n    log(f"[sft] val losses {hist[\'val\']}")\n\n    cell("sft save")\n    od = out_dir(args.name)\n    model.save_pretrained(od / "adapter")\n    tok.save_pretrained(od / "adapter")\n    save_json(od / "history.json", hist)\n    if args.merge:\n        mdir = Path(args.merge_dir) if args.merge_dir else od / "merged"\n        merged = model.merge_and_unload()\n        merged.save_pretrained(mdir, safe_serialization=True)\n        tok.save_pretrained(mdir)\n        log(f"[sft] merged model -> {mdir}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n',
    'eval_router.py': '"""prompt_router_v1 planner: score Qwen3-0.6B (base and + LoRA) on router_test with router_metrics.score.\n\n    python -u eval_router.py --base Qwen/Qwen3-0.6B --adapter out/router-0p6b/adapter\n\nGreedy, thinking off, JSON parsed by code (an unparsed answer counts as a refusal). Writes out/router_eval/<tag>.json.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport re\nimport sys\nimport time\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom common import cell, jl, log, out_dir, save_json, wjl  # noqa: E402\nfrom router_metrics import score  # noqa: E402\n\n\ndef parse(text: str) -> dict | None:\n    text = re.sub(r"<think>.*?</think>", "", text or "", flags=re.S)\n    m = re.search(r"\\{.*\\}", text, flags=re.S)\n    if not m:\n        return None\n    try:\n        d = json.loads(m.group(0))\n    except json.JSONDecodeError:\n        return None\n    if not isinstance(d, dict) or not isinstance(d.get("in_scope"), bool):\n        return None\n    d["tasks"] = [t for t in d.get("tasks") or [] if isinstance(t, str)]\n    d["slots"] = d.get("slots") if isinstance(d.get("slots"), dict) else {}\n    return d\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--base", default="Qwen/Qwen3-0.6B")\n    ap.add_argument("--adapter", default="")\n    ap.add_argument("--tag", default="router-0p6b")\n    a = ap.parse_args()\n    cell("router eval load")\n    te = jl("router_test.jsonl")\n    from transformers import AutoTokenizer\n    from vllm import LLM, SamplingParams\n    tok = AutoTokenizer.from_pretrained(a.base)\n    prompts = [tok.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=True, enable_thinking=False)\n               for r in te]\n    kw = dict(enable_lora=True, max_lora_rank=64) if a.adapter else {}\n    llm = LLM(model=a.base, max_model_len=2048, gpu_memory_utilization=0.85, seed=0, **kw)\n    systems = [("base", None)]\n    if a.adapter:\n        from vllm.lora.request import LoRARequest\n        systems.append(("lora", LoRARequest("router", 1, a.adapter)))\n    od, res = out_dir("router_eval"), {"base": a.base, "adapter": a.adapter, "systems": {}}\n    for name, lora in systems:\n        cell(f"router eval generate {name}")\n        t0 = time.time()\n        outs = llm.generate(prompts, SamplingParams(temperature=0.0, max_tokens=300), lora_request=lora)\n        preds = [parse(o.outputs[0].text) for o in outs]\n        s = score(te, preds)\n        s["seconds"] = round(time.time() - t0, 1)\n        res["systems"][name] = s\n        wjl(od / f"{a.tag}_{name}_answers.jsonl", [{"id": r["id"], "text": o.outputs[0].text} for r, o in zip(te, outs)])\n        log(f"[router] {name}: OOS recall {s[\'oos_recall\']} {s[\'oos_recall_ci95\']} · false refusal {s[\'false_refusal\']} · "\n            f"task macro-F1 {s[\'task_macro_f1\']} · claim EM {s[\'slot_claim_exact\']} · unparsed {s[\'unparsed\']}")\n    save_json(od / f"{a.tag}.json", res)\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n',
    'export_gguf.sh': '#!/usr/bin/env bash\n# Merged HF model -> GGUF Q4_K_M + Ollama Modelfile, for the offline laptop (Ollama).\n#   bash export_gguf.sh /tmp/explain-4b-merged out/explain-4b-gguf greenscan-explain-4b\nset -euo pipefail\nMERGED="$1"; OUT="$2"; NAME="${3:-greenscan-model}"\necho "[cell] gguf build llama.cpp"\nmkdir -p "$OUT"\n[ -d /tmp/llama.cpp ] || git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp\npip install -q /tmp/llama.cpp/gguf-py sentencepiece protobuf\ncmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF > /tmp/cmake.log\ncmake --build /tmp/llama.cpp/build --target llama-quantize -j "$(nproc)" > /tmp/build.log\necho "[cell] gguf convert f16"\npython -u /tmp/llama.cpp/convert_hf_to_gguf.py "$MERGED" --outfile /tmp/model-f16.gguf --outtype f16\necho "[cell] gguf quantize Q4_K_M"\n/tmp/llama.cpp/build/bin/llama-quantize /tmp/model-f16.gguf "$OUT/$NAME-q4_k_m.gguf" Q4_K_M\nrm -f /tmp/model-f16.gguf\ncat > "$OUT/Modelfile" <<EOF\nFROM ./$NAME-q4_k_m.gguf\nPARAMETER temperature 0\nPARAMETER num_ctx 4096\nEOF\nls -la "$OUT"\necho "[gguf] done: ollama create $NAME -f $OUT/Modelfile"\n',
}
for name, text in FILES.items():
    open(name, 'w', encoding='utf-8').write(text)
    print('wrote', name, len(text), flush=True)


In [ ]:
print('[cell] run 1: python -u train_router_gate.py', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('python -u train_router_gate.py')


In [ ]:
print('[cell] run 2: python -u train_llm_sft.py --base Qwen/Qwen3-0.6B --train ro', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('python -u train_llm_sft.py --base Qwen/Qwen3-0.6B --train router_train.jsonl --name router-0p6b --epochs 3 --lr 2e-4 --dtype fp16 --max-len 1024 --merge --merge-dir /tmp/router-merged')


In [ ]:
print('[cell] run 3: python -u eval_router.py --base Qwen/Qwen3-0.6B --adapter ou', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('python -u eval_router.py --base Qwen/Qwen3-0.6B --adapter out/router-0p6b/adapter')


In [ ]:
print('[cell] run 4: bash export_gguf.sh /tmp/router-merged out/router-0p6b-gguf ', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('bash export_gguf.sh /tmp/router-merged out/router-0p6b-gguf greenscan-router-0p6b')


In [ ]:
print("[cell] done", flush=True)
print("ALL CELLS OK", flush=True)